In [ ]:
# basic multiprocess program

from multiprocessing import Process
import os

def task():
    print("Currently Executing Child Process")
    print("This process has it's own instance of the GIL")
    print(f"this process: {os.getpid()}")
    print("Executing Main Process")
    print("Creating Child Process")

if __name__ == '__main__':
    print(os.getpid())
    myProcess = Process(target=task)
    print(os.getpid())
    myProcess.start()
    print(os.getpid())
    myProcess.join()
    print(os.getpid())
    print("Child Process has terminated, terminating main process")

### Starting a process

There's 3 ways to do it, atleast in the multiprocessing module

1. Spawn
2. Fork
3. Forkserver

### Daemon processes in python

Daemon means the same as it meant in context of threads..

A daemon process exits when the main process exits, or if it finishes its task function, whichever comes first.

In [ ]:
from multiprocessing import Process, current_process
import time

def daemonProcess():
    print("Starting my Daemon Process")
    print("Daemon process started: {}".format(current_process())) # current_process() fetches the process object. if you only wanted pid, you can do current_process().pid or os.getpid()
    time.sleep(3)
    print("Daemon process terminating")

if __name__ == '__main__':
    print("Main process: {}".format(current_process()))
    myProcess = Process(target=daemonProcess, daemon=True)
    myProcess.start()
    print("We can carry on as per usual and our daemon will continue to execute")
    time.sleep(11) # remove this timer, you'll notice that the daemon process did not get time to run its target function completely.

**NOTE**: It should be noted that you cannot create child processes from daemon
processes; if you try this, it will fail on calling process.start().

In [ ]:
# you can name your processes just like your threads

import multiprocessing

def myProcess():
    print("{} Just performed X".format(multiprocessing.current_process().name))

def main():
    childProcess = multiprocessing.Process(target=myProcess, name='MyAwesome-Process')
    childProcess.start()
    childProcess.join()

if __name__ == '__main__':
    main()

### Terminating Processes

You want to terminate them to clean up after your work, especially in production environments...

```python
process.terminate()
```

In [ ]:
import multiprocessing
import time

def task():
    current_process = multiprocessing.current_process()
    print("Child Process PID: {}".format(current_process.pid))
    for i in range(1,20):
        print(i)
        time.sleep(1)
    print("Child process done!")

def main():
    print("Main process PID: {}".format(multiprocessing.current_process().pid))
    myProcess = multiprocessing.Process(target=task)
    myProcess.start()
    time.sleep(3)
    print("Terminating Child Process midway through its execution")
    # myProcess.join()
    myProcess.terminate()
    print("Child Process Successfully terminated")

if __name__ == '__main__':
    main()

# NOTE: if you can afford to let your processes finish naturally (most times you might not), then terminate() isn't required, just wait for the processes to join()... However, in cases where you want to enforce a timeout, this is a good way to write it - 

myprocess.join(timeout=5) # join all processes that finish within 5s, let the others continue running

if myProcess.is_alive():  # Still running?
    myProcess.terminate()  # Actually kill it                               
    myProcess.join()       # Wait for termination to complete. You have to wait for them to join() even here because terminate() is async, it doesn't take effect immediately.

### Subclassing Processes

You may want to do this to incorporate your desired behaviour in the process.

In [ ]:
from multiprocessing import Process, current_process
class MyProcess(Process):
    def __init__(self):
        super(MyProcess, self).__init__()

    def run(self):
        print("Child Process PID: {}".format(current_process().pid))

def main():
    print("Main Process PID: {}".format(current_process().pid))
    myProcess = MyProcess()
    myProcess.start()
    myProcess.join()

if __name__ == '__main__':
    main() 

### Pools

You use these when you want a number of worker processes to do some job for you.

### why pools when you had processpoolexecutor?

Apparently processpoolexecutor abstract a lott at the cost of being easy to use. While they still will achieve 99% of what you want, it's good to know pools too for cases where you need fine-grained control

If using pools, use them in context managers only, that's the best way to do it.

In [ ]:
from multiprocessing import Pool

def task(n):
    print(n)
    return 2 * n

def main():
    with Pool(4) as p:
        print(p.map(task, [2,3,4]))
        
if __name__ == '__main__':
    main()

### Submitting Tasks to Pool

### 1. apply() method

In [ ]:
from multiprocessing import Pool
import time

def myTask(n):
    time.sleep(n/2)
    return n*2

def main():
    with Pool(4) as p:
        print(p.apply(myTask, (4,)))
        print(p.apply(myTask, (3,)))
        print(p.apply(myTask, (2,)))
        print(p.apply(myTask, (1,)))

if __name__ == '__main__':
    main()

# apply() blocks until result returns, so it's using a multi-process program to do things in a single-threaded manner really... Not useful if you want jobs done parallely.

### 2. apply_async()

for parallel task execution

In [ ]:
from multiprocessing import Pool
import os
import time

def myTask(n):
    time.sleep(1)
    print("Task processed by Process {}".format(os.getpid()))
    return n*2

def main():
    print("apply_async")
    with Pool(4) as p:
        tasks = []
        for i in range(4):
            task = p.apply_async(func=myTask, args=(i,))
            tasks.append(task)

        for task in tasks:
            task.wait()
            print("Result: {}".format(task.get()))

if __name__ == '__main__':
    main()

### 3. map()

use this to map a list of input values to target function. Each input value - target fn pair will then have its own process running it.

Alas, this uses the apply() function underneath, use map_async()

In [ ]:
from multiprocessing import Pool
import time
import os

def myTask(n):
    time.sleep(1)
    print("Task processed by Process {}".format(os.getpid()))
    return n*2

def main():
    print("mapping array to pool")
    with Pool(4) as p:
        print(p.map(myTask, [0,1,2,3]))

if __name__ == '__main__':
    main()

### 4. map_async()

In [ ]:
from multiprocessing import Pool
import time

def myTask(n):
    time.sleep(1)
    print("Task processed by Process {}".format(os.getpid()))
    return n*2

def main():
    with Pool(4) as p:
        print(p.map_async(myTask, [4,3,2,1]).get())

if __name__ == '__main__':
    main()

### 5. imap()

Does the same thing as map(), but returns an iterable instead of a list.

It is slower than map() apparently, so i'm not sure where i'd get to use it.

In [ ]:
from multiprocessing import Pool
import time
import os

def myTask(n):
    time.sleep(1)
    print("Task processed by Process {}".format(os.getpid()))
    return n*2

def main():
    with Pool(4) as p:
        for iter in p.imap(myTask, [1,3,2,1]):
            print(iter)

if __name__ == '__main__':
    main()

### 6. imap_unordered()

Does the same thing as imap(), but returns results in order of completion, which by itself is random..

In [ ]:
from multiprocessing import Pool
import time
import os

def myTask(n):
    time.sleep(1)
    print("Task processed by Process {}".format(os.getpid()))
    return n*2
    
def main():
    with Pool(4) as p:
        for iter in p.imap_unordered(myTask, [1,3,2,1]):
            print(iter)

if __name__ == '__main__':
    main()

### 7. starmap()

Say you have a function 

```python
def task(a, b, c):
``` 

and a list of tuples 
```python
[(1, 2, 3), (4, 5, 6)]
```

When you use 

```python
pool.map(task, [(1, 2, 3), (4, 5, 6)])
```

the map method iterates through the list and passes each item directly to the function—so it calls 

```python
task((1, 2, 3))
```

which means the entire tuple becomes a single argument. But your function expects 3 separate arguments, so it crashes. 

```python
pool.starmap(task, [(1, 2, 3), (4, 5, 6)])
```

solves this by unpacking each tuple using the * operator before passing it to the function—so it calls 

```python
task(1, 2, 3)
```

instead. That's why it's called "starmap": it's map with star unpacking. The star does the unpacking, allowing each tuple in your iterable to be spread into the multiple arguments your function needs.

In [ ]:
from multiprocessing import Pool
import time

def myTask(x, y):
    time.sleep(x/2)
    return y*2

def main():
    with Pool(4) as p:
        print(p.starmap(myTask, [(4,3),(2,1)]))

if __name__ == '__main__':
    main()

### 8. starmap_async()

probably the best one..

In [ ]:
from multiprocessing import Pool
import time

def myTask(x, y):
    time.sleep(x/2)
    return y*2

def main():
    with Pool(4) as p:
        result = p.starmap_async(myTask, [(4,3),(2,1)])
        # do other tasks
        print(result.get())

if __name__ == '__main__':
    main()

### Maxtasksperchild

When you run a program that does a lot of work, the worker processes handling that work can sometimes accumulate "baggage" over time — things like memory that wasn't properly released, or other small inefficiencies that build up. Eventually, this can slow things down or waste resources.

**The solution: worker recycling**

Applications like Apache (a web server) already solve this problem by automatically retiring a worker process after it has handled a certain amount of work, then spinning up a fresh one to replace it. Think of it like a shift change — after a worker completes X number of tasks, they clock out and a fresh worker clocks in.

**How this applies to Python's multiprocessing Pool**

Python's `multiprocessing.Pool` lets you do the same thing using the `maxtasksperchild` parameter. When you create a Pool, you can say: *"after any given worker process completes N tasks, kill it and replace it with a new one."*

For example:
```python
pool = multiprocessing.Pool(processes=4, maxtasksperchild=5)
```

This means each worker process will handle up to 5 tasks, then be recycled — freeing up any memory or resources it was holding — and a fresh worker takes its place.

**Why does this matter?**

It's essentially a safeguard against long-running programs gradually degrading in performance. By periodically refreshing your workers, you keep things clean and efficient, similar to restarting an app to clear it out. The work still gets done, just by a fresh process instead.

Let's demo it by using it with the `starmap_async` and added `maxtasksperchild` to it to demonstrate this recycling behavior in action.

In [ ]:
from multiprocessing import Pool
import time
import os

def myTask(x, y):
    print("{} Executed my task".format(os.getpid()))
    return y*2

def main():
    with Pool(processes=2, maxtasksperchild=2) as p:
        result1 = p.starmap_async(myTask, [(4,3),(2,1), (3,2), (5,1)])
        result2 = p.starmap_async(myTask, [(4,3),(2,1), (3,2), (2,3)])
        print(result1.get())
        print(result2.get())

if __name__ == '__main__':
    main()

# you'll notice the same process id doesn't appear more than twice, that's because after 2 tasks, the worker process is terminated and replace by another worker.

### Communication / Synchronization between Worker Processes

There's 4 main ways for this - 

1. Queues: We covered how to share exceptions between threads (child threads to main thread) using queues. It's the same concept here mostly.

2. Pipes: new concept

3. Manager: another new concept i think

4. Ctypes: new.

### Pipes

they're queues for passing information between processes. Most major OS's use these, and we've imitated that functionality to work in python. There's 2 types - 

Anonymous pipe and named pipe.

In [ ]:
# this is supposed to be a tutorial on how to use a pipe (in a very basic manner mind you), but it it didn't work out of the box... reason in the markdown cell following this cell.

import os, sys
from multiprocessing import Process
from typing import TextIO

class ChildProcess(Process):
    def __init__(self, pipein):
        super(ChildProcess, self).__init__()
        self.pipein: TextIO = pipein

    def run(self):
        print("Attempting to pipein to pipe")
        self.pipein = os.fdopen(self.pipein, 'w')
        self.pipein.write("My Name is Ketan")
        self.pipein.close()

def main():
    pipeout, pipein = os.pipe()
    child = ChildProcess(pipein)
    child.start()
    child.join()
    os.close(pipein)
    pipeout = os.fdopen(pipeout)
    pipeContent = pipeout.read()
    print("Pipe: {}".format(pipeContent))

if __name__ == '__main__':
    multiprocessing.set_start_method('fork')  # Tutorial did not have this line, perhaps he had a linux machine... I need to pass this in order for it to work. It's also mentioned that this may introduce bugs, so try to use multiprocessing.Pipe() instead of os.pipe(), since it's cross-platform.
    main()

## The Problem

When using `multiprocessing`, Python **pickles and transfers** the `ChildProcess` object to the child process. Raw OS file descriptors (integers) **don't survive this transfer** on macOS/Windows — the integer gets copied, but the underlying file descriptor is closed/invalid in the child's context.

This is a known platform difference: on **Linux**, `fork()` duplicates file descriptors automatically, so the tutorial likely worked for the author. On **macOS** (which you're using), `multiprocessing` defaults to `spawn`, which doesn't inherit file descriptors.

## The Fix

Use `multiprocessing.Pipe()` instead of `os.pipe()`, since it's designed to work across processes properly. Or, if you want to keep `os.pipe()`, explicitly pass it via `initializer` or use `reduction.ForkingPickler`.

The cleanest fix:

```python
import os
import multiprocessing

class ChildProcess(multiprocessing.Process):
    def __init__(self, pipein):
        super(ChildProcess, self).__init__()
        self.pipein = pipein

    def run(self):
        print("Attempting to write to pipe")
        self.pipein.write("My Name is Elliot")
        self.pipein.close()

def main():
    pipeout, pipein = multiprocessing.Pipe(duplex=False)  # one-way pipe
    
    # Convert to a writable connection object before passing
    child = ChildProcess(pipein)
    child.start()
    child.join()

    pipein.close()
    pipeContent = pipeout.recv()  # use recv() instead of read()
    print("Pipe: {}".format(pipeContent))

if __name__ == '__main__':
    main()
```

## Why This Works

| | `os.pipe()` | `multiprocessing.Pipe()` |
|---|---|---|
| Returns | Raw int file descriptors | `Connection` objects |
| Survives `spawn` | ❌ No | ✅ Yes |
| Cross-platform | ❌ Fragile | ✅ Reliable |
| Read method | `.read()` | `.recv()` |

## If You Must Use `os.pipe()`

Force Linux-style `fork` at the top of your script (macOS only, not recommended for production):

```python
multiprocessing.set_start_method('fork')  # add this before main()
```

This makes the tutorial's original code work, but `fork` on macOS can cause subtle bugs with certain libraries (like those using threads internally).

-------------------------------------------

I barely understood anything in the above text about pickling, and file descriptors... Ig I'll need separate reading on that.

In [ ]:
# A version using multiprocessing.Pipe() instead, since it's a safer option. I'll use this only most prob

import os
import multiprocessing

class ChildProcess(multiprocessing.Process):
    def __init__(self, pipein):
        super(ChildProcess, self).__init__()
        self.pipein = pipein

    def run(self):
        print("Attempting to write to pipe")
        self.pipein.send("My Name is Ketan")
        self.pipein.close()

def main():
    pipeout, pipein = multiprocessing.Pipe(duplex=False)  # one-way pipe
    
    # Convert to a writable connection object before passing
    child = ChildProcess(pipein)
    child.start()
    child.join()

    pipein.close()
    pipeContent = pipeout.recv()  # use recv() instead of read()
    print("Pipe: {}".format(pipeContent))

if __name__ == '__main__':
    main()

# NOTE: I couldn't demonstrate this for some reason, but I've been told by Claude that you can safely do pipeout.recv() ONLY when all of the write ends are closed (every copy of pipein has closed). it's kind of like an 'over' command in a walkie-talkie convo.

# so if a process has nothing to say (like the main process here, which only created the pipe, but didn't actually have anything to input to the pipe), close the write-end of the pipe as soon as you've created the child.

### Handling Exceptions

You can use pipes for inter-process communication too, which by extension means you can propagate exceptions using pipes.

In [ ]:
from multiprocessing import Process, Pipe
import os, sys
import traceback

class MyProcess(Process):
    def __init__(self, pipein):
        super(MyProcess, self).__init__()
        self.pipein = pipein

    # the run method writes an exception to a pipe.
    def run(self):
        try:
            raise Exception("This broke stuff")
        except Exception as e:
            # except_type, except_value, tb = sys.exc_info()
            # self.pipein.send(except_value) # you could send just the exception value, extracted in the above stmt, or just pass the whole exception, which is normally what you'd do.
            self.pipein.send(e)
            self.pipein.close()

def main():
    pipeout, pipein = Pipe(duplex=False)
    childProcess = MyProcess(pipein)
    childProcess.start()
    childProcess.join()
    pipein.close()
    pipeContent = pipeout.recv()
    print("Exception: {}".format(pipeContent))

if __name__ == '__main__':
    main()

### when to use pipes? 

Pipes are great for inter-process communication, but only when it's only 2 processes partaking in communication... It has very less overhead, so using them makes sense from performance pov too...

But what if you want to implement a means of communication that all of your worker processes (>10) can access?

You use `Managers`

### Multiprocessing Managers

It's a class via which we can share stuff such as lists / dicts across processes... It's the prime candidate for true inter-process communication. It does this by starting a server process, and handing out proxies to all other processes...

### Namespaces

Managers use namespaces often.. from what i understand, a namespace is like a scratchpad for all of your processes to dump / access info from... it has no methods, just attributes you can access and write to and read from.

by claude, a namespace is a flexible attribute bag you can attach arbitrary variables to, useful when a list or dict is too rigid for what you need to share


In [ ]:
# using a namespace

import multiprocessing as mp
import queue

def task(ns):
    # Update values within our namespace
    print(ns.x)
    ns.x = 2

def main():
    manager = mp.Manager()
    ns = manager.Namespace()
    ns.x = 1
    ns.name = "ketan"
    print(ns)
    process = mp.Process(target=task, args=(ns,))
    process.start()
    process.join()
    print(ns)

if __name__ == '__main__':
    main()

# pretty nifty... idk why processes need itna freedom for writing / accessing values tbh...

### Queues

We've covered communication using queues when we worked with multi-threaded programs... But let's have an example of usage with processes nonetheless.

In [ ]:
import multiprocessing
from queue import Queue
import time

def myTask(queue: Queue):
    value = queue.get()
    print("Process {} Popped {} from the shared Queue".format(multiprocessing.current_process().pid, value))
    queue.task_done()

def main():
    m = multiprocessing.Manager()
    sharedQueue = m.Queue()
    sharedQueue.put(2)
    sharedQueue.put(3)
    sharedQueue.put(4)

    process1 = multiprocessing.Process(target=myTask, args=(sharedQueue,))
    process1.start()
    process1.join()
    process2 = multiprocessing.Process(target=myTask, args=(sharedQueue,))
    process2.start()
    process2.join()
    process3 = multiprocessing.Process(target=myTask, args=(sharedQueue,))
    process3.start()
    process3.join()

if __name__ == '__main__':
    main()

### Listeners and Clients

Consider this as your first foray into distributed systems... Listeners and clients handle IPC when the processes do NOT have the same parent. This enables IPC for processes running far away from each other, on different machines even...

There's the barebones 'connection' sub-module within multiprocessing that can handle this, or full-fledged messaging libraries / brokers such as zeromq, rabbitmq or redis that could handle this for you.

We'll look into the barebones, self-implemented version of listeners and clients.

In [ ]:
# listener / client scripts in their own dedicated .py files.

# NOTE: listener is the sender, client is the receiver